# Dashboard (Phase 5)

Subscribes to all agent topics and renders live state using anymap-ts only.

## Quick run order (easy mode)
1. Run cells 1-4 in this notebook first (connect + subscribe).
2. Run publish cells in `agent_roadwork`, `agent_cars`, and `agent_monitor`.
3. Re-run cell 5 here to see latest counts.
4. Only disconnect when you are completely finished.

In [21]:
# Cell purpose: Load config, connect MQTT, and initialize dashboard state containers.
import json
import uuid

from simulated_city import mqtt
from simulated_city.config import load_config
from simulated_city.topics import (
    cars_reroute_topic,
    cars_telemetry_topic,
    roadwork_events_topic,
    traffic_congestion_topic,
)

config = load_config()
phase1 = config.simulation.car_rerouting_phase1 if config.simulation else None
if phase1 is None:
    raise ValueError("Missing simulation.car_rerouting_phase1 in config.yaml")

dashboard_consumer_running = False
previous_task = globals().get("dashboard_consumer_task")
if previous_task is not None and not previous_task.done():
    previous_task.cancel()

previous_client = globals().get("client")
if previous_client is not None:
    try:
        previous_client.loop_stop()
        previous_client.disconnect()
    except Exception:
        pass

client_suffix = f"dashboard-phase5-{uuid.uuid4().hex[:6]}"
client = mqtt.connect_mqtt(config.mqtt, client_id_suffix=client_suffix)
telemetry_topic = cars_telemetry_topic(config.mqtt.base_topic)
reroute_topic = cars_reroute_topic(config.mqtt.base_topic)
roadwork_topic = roadwork_events_topic(config.mqtt.base_topic)
congestion_topic = traffic_congestion_topic(config.mqtt.base_topic)

latest_telemetry_by_car = {}
latest_reroute_by_car = {}
latest_roadwork = {"active": False, "blocked_segment_ids": []}
latest_congestion = {"congested_segment_ids": [], "cars_per_segment": {}, "tick": 0}

print(f"Connected MQTT target: {config.mqtt.host}:{config.mqtt.port}")
print(f"Dashboard client suffix: {client_suffix}")
print(
    f"Dashboard topics: telemetry={telemetry_topic}, reroute={reroute_topic}, "
    f"roadwork={roadwork_topic}, congestion={congestion_topic}"
)

Connected MQTT target: 127.0.0.1:1883
Dashboard client suffix: dashboard-phase5-75ec95
Dashboard topics: telemetry=simulated-city/city/cars/telemetry, reroute=simulated-city/city/cars/reroute, roadwork=simulated-city/city/roadwork/events, congestion=simulated-city/city/traffic/congestion


In [22]:
# Cell purpose: Create the anymap-ts map widget and render initial empty view.
from anymap_ts import Map

dashboard_map = Map(
    center=(12.5683, 55.6761),
    zoom=13.4,
    height="640px",
)
dashboard_map.add_basemap("OpenStreetMap.Mapnik")
dashboard_map

In [23]:
# Cell purpose: Subscribe to MQTT topics and update map overlays from incoming events.
import asyncio
import hashlib
import math
from queue import Empty, Queue

from simulated_city.maplibre_live import car_popup_text, resolve_node_lnglat, resolve_segment_lnglat

active_markers = set()
latest_telemetry_tick = 0
incoming_events = Queue()

def _car_marker_jitter(car_id: str) -> tuple[float, float]:
    digest = int(hashlib.sha256(car_id.encode("utf-8")).hexdigest()[:8], 16)
    angle_rad = math.radians(digest % 360)
    radius_m = 5.0 + float(digest % 5)
    lat_offset = (radius_m / 111_111.0) * math.sin(angle_rad)
    lng_offset = (radius_m / (111_111.0 * math.cos(math.radians(55.6761)))) * math.cos(angle_rad)
    return lng_offset, lat_offset

def redraw_dashboard():
    global active_markers
    for marker_id in list(active_markers):
        try:
            dashboard_map.remove_marker(marker_id)
        except Exception:
            pass
    active_markers = set()

    for car_id, event in sorted(latest_telemetry_by_car.items()):
        marker_id = f"car-{car_id}"
        if "current_lng" in event and "current_lat" in event:
            lng = float(event["current_lng"])
            lat = float(event["current_lat"])
        else:
            node = str(event.get("current_node", ""))
            if not node:
                continue
            try:
                lng, lat = resolve_node_lnglat(node)
            except KeyError:
                continue
            jitter_lng, jitter_lat = _car_marker_jitter(car_id)
            lng += jitter_lng
            lat += jitter_lat

        dashboard_map.add_marker(
            lng,
            lat,
            name=marker_id,
            color="#3388ff",
            popup=car_popup_text(event),
        )
        active_markers.add(marker_id)

    for segment_id in latest_roadwork.get("blocked_segment_ids", []):
        try:
            lng, lat = resolve_segment_lnglat(int(segment_id), segment_node_pairs=phase1.segment_node_pairs)
        except KeyError:
            continue
        marker_id = f"roadwork-{segment_id}"
        dashboard_map.add_marker(
            lng,
            lat,
            name=marker_id,
            color="#d32f2f",
            popup=f"roadwork segment={segment_id}",
        )
        active_markers.add(marker_id)

    for segment_id in latest_congestion.get("congested_segment_ids", []):
        try:
            lng, lat = resolve_segment_lnglat(int(segment_id), segment_node_pairs=phase1.segment_node_pairs)
        except KeyError:
            continue
        marker_id = f"congestion-{segment_id}"
        dashboard_map.add_marker(
            lng,
            lat,
            name=marker_id,
            color="#ff9800",
            popup=f"congestion segment={segment_id}",
        )
        active_markers.add(marker_id)

def _apply_event(topic: str, payload: dict):
    global latest_telemetry_tick
    if topic == telemetry_topic:
        latest_telemetry_by_car[str(payload.get("car_id", "unknown"))] = payload
        latest_telemetry_tick = max(latest_telemetry_tick, int(payload.get("tick", 0)))
    elif topic == reroute_topic:
        latest_reroute_by_car[str(payload.get("car_id", "unknown"))] = payload
    elif topic == roadwork_topic:
        latest_roadwork.update(payload)
    elif topic == congestion_topic:
        latest_congestion.update(payload)

def drain_dashboard_events(max_events: int = 500) -> int:
    processed = 0
    while processed < max_events:
        try:
            topic, payload = incoming_events.get_nowait()
        except Empty:
            break
        _apply_event(topic, payload)
        processed += 1

    if processed > 0:
        redraw_dashboard()
    return processed

dashboard_consumer_running = True
dashboard_consumer_task = None

async def _dashboard_consumer_loop(poll_interval_s: float = 0.1):
    while dashboard_consumer_running:
        drain_dashboard_events()
        await asyncio.sleep(poll_interval_s)

def _on_message(client_obj, userdata, msg):
    payload = json.loads(msg.payload.decode("utf-8"))
    incoming_events.put((msg.topic, payload))

def _subscribe_dashboard_topics():
    client.subscribe(telemetry_topic)
    client.subscribe(reroute_topic)
    client.subscribe(roadwork_topic)
    client.subscribe(congestion_topic)

_previous_on_connect = getattr(client, "on_connect", None)
def _on_connect(client_obj, userdata, flags, reason_code, properties=None):
    if callable(_previous_on_connect):
        _previous_on_connect(client_obj, userdata, flags, reason_code, properties)
    _subscribe_dashboard_topics()

client.on_connect = _on_connect
client.on_message = _on_message
_subscribe_dashboard_topics()

try:
    dashboard_consumer_task = asyncio.create_task(_dashboard_consumer_loop())
    print("Dashboard subscriptions active with async queue consumer.")
except RuntimeError:
    print("Dashboard subscriptions active. Auto-consumer unavailable; call drain_dashboard_events() manually.")

Dashboard subscriptions active with async queue consumer.


In [24]:
# Cell purpose: Show a concise textual snapshot of the latest dashboard state.
processed_now = drain_dashboard_events()
redraw_dashboard()
expected_cars = int(phase1.car_count)
tracked_cars = len(latest_telemetry_by_car)

print(f"Events drained now: {processed_now}")
print(f"Cars tracked: {tracked_cars}/{expected_cars}")
print(f"Latest telemetry tick seen: {latest_telemetry_tick}")
print(f"Reroute events tracked: {len(latest_reroute_by_car)}")
print(f"Roadwork active: {latest_roadwork.get('active', False)} blocked={latest_roadwork.get('blocked_segment_ids', [])}")
print(
    f"Congested segments: {latest_congestion.get('congested_segment_ids', [])} "
    f"(tick={latest_congestion.get('tick', 0)})"
)

if tracked_cars < expected_cars:
    print("Tip: Start dashboard first, then run agent publisher cells while dashboard stays connected.")
if latest_telemetry_tick <= 5:
    print("Tip: Agent cars streams multiple ticks of movement; re-run its publish cell to continue the live feed.")
if not latest_roadwork.get('blocked_segment_ids') and processed_now == 0:
    print("Tip: Run roadwork publisher after dashboard subscriptions to populate live roadwork markers.")

Events drained now: 17
Cars tracked: 10/10
Latest telemetry tick seen: 66
Reroute events tracked: 0
Roadwork active: False blocked=[44105317, 733901267]
Congested segments: [] (tick=0)


In [ ]:
# Cell purpose: Disconnect dashboard MQTT client only when you explicitly choose to stop.
# Run All safe default: keep subscription alive for incoming agent events.
disconnect_now = False

if disconnect_now:
    dashboard_consumer_running = False
    if dashboard_consumer_task is not None and not dashboard_consumer_task.done():
        dashboard_consumer_task.cancel()
    client.loop_stop()
    client.disconnect()
    print("Disconnected MQTT client and stopped dashboard consumer.")
else:
    print("Dashboard still connected. Set disconnect_now = True and re-run this cell to disconnect.")